# Rainfall Analysis and Planting Advisory

**PLP Data Analysis Assignment 2**

This notebook cleans monthly rainfall records for five Kenyan counties, analyses rainfall patterns with NumPy, and produces a simple planting-window advisory. Rainfall values are in millimetres (mm).

## 1. Import libraries and define the data

The dataset contains four already-clean county rows and one manually entered Machakos row that requires cleaning.

In [1]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = np.array([
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

print("Number of counties:", len(counties))
print("Number of months:", len(months))

Number of counties: 5
Number of months: 12


## 2. Clean the Machakos rainfall data

The cleaning function converts valid text values to integers. If conversion fails, `ValueError` is caught and the value is replaced with the rounded mean of the valid Machakos readings. Blank text and `N/A` are therefore treated as invalid values.

In [2]:
def clean_rainfall(raw_values):
    valid_values = []
    for value in raw_values:
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            pass

    replacement = round(np.mean(valid_values))
    cleaned_values = []
    imputed_months = []

    for month, value in zip(months, raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(replacement)
            imputed_months.append(month)

    return cleaned_values, imputed_months, replacement


machakos_clean, imputed_months, imputed_value = clean_rainfall(machakos_raw)

print("Valid-reading average (rounded):", imputed_value, "mm")
print("Imputed months:", ", ".join(imputed_months))
print("Cleaned Machakos row:", machakos_clean)

Valid-reading average (rounded): 68 mm
Imputed months: April, September
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combine the counties into a NumPy array

The four clean rows and the cleaned Machakos row are combined into one two-dimensional array. The expected dimensions are five counties by twelve months.

In [3]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Rainfall array:")
print(rainfall)
print("Array shape:", rainfall.shape)
print("Array dtype:", rainfall.dtype)

Rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array dtype: int64


## 4. Annual rainfall totals and monthly averages

`sum(axis=1)` adds the twelve monthly values for each county. `mean(axis=0)` calculates the average for each month across the five counties.

In [4]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)

print("Annual rainfall totals (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall by month across all counties (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual rainfall totals (mm):
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall by month across all counties (mm):
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


## 5. Wettest month for each county

`np.argmax(axis=1)` returns the column position of the highest monthly rainfall in each county's row. `zip()` pairs each county with its wettest month.

In [5]:
wettest_indices = np.argmax(rainfall, axis=1)
wettest_months = months[wettest_indices]
wettest_amounts = rainfall[np.arange(len(counties)), wettest_indices]

print("Wettest month by county:")
for county, month, amount in zip(counties, wettest_months, wettest_amounts):
    print(f"{county}: {month} ({amount} mm)")

Wettest month by county:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


## 6. Count dry months

A dry month is defined in this assignment as a month with rainfall strictly below 50 mm. The Boolean condition creates a mask, and `sum(axis=1)` counts the `True` values in each county.

In [6]:
dry_month_mask = rainfall < 50
dry_month_counts = dry_month_mask.sum(axis=1)

print("Number of dry months (below 50 mm):")
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Number of dry months (below 50 mm):
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## Planting Advisory

For this simple advisory, a month with **at least 100 mm** of recorded rainfall is flagged as a potential planting window. This is a screening rule for the assignment, not a complete agronomic recommendation. Farmers should also consider soil moisture, rainfall onset and distribution, crop type, seed variety, and local agricultural extension advice.

In [7]:
planting_threshold_mm = 100
planting_windows = [
    ", ".join(months[row >= planting_threshold_mm])
    if np.any(row >= planting_threshold_mm)
    else "No month reaches 100 mm"
    for row in rainfall
]

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

Nairobi: April, May, November, December
Nakuru: April, May, August
Kisumu: March, April, May, November
Mombasa: April, May, June, November
Machakos: May, November


## Data Quality Notes

- Machakos contained two invalid entries: `N/A` for April and a blank value for September. Both were replaced with the rounded mean of the ten valid readings.
- The imputed value is **68 mm**, based on the available Machakos readings. Imputation makes the array complete but introduces uncertainty into the affected months and the annual total.
- The supplied readings are monthly totals and do not show within-month rainfall distribution, start-of-season dates, forecast conditions, soil type, or crop water requirements.
- The 100 mm planting-window threshold is a transparent assignment heuristic. It should not be treated as a guaranteed planting date or as a substitute for local agricultural guidance.

## 7. Export the county summary to CSV

The CSV contains exactly one row per county and the five columns requested in the assignment.

In [8]:
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        "county", "annual_total_mm", "wettest_month",
        "dry_months", "planting_window"
    ])
    writer.writerows(zip(
        counties, annual_totals, wettest_months,
        dry_month_counts, planting_windows
    ))

print("rainfall_summary.csv has been created.")

rainfall_summary.csv has been created.
